---
title: "5.4 Checking AI coding assistance"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/05-workflow/04-checking-ai.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/05-workflow/04-checking-ai.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If the GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. Each notebook starts afresh.

**Time:** about 35–40 minutes, including practice. **Before you begin:** Complete [5.3 Public data by code](03-public-data.ipynb). No AI account is needed for this lesson.

## You will learn

- Turn an analysis request into observable checks before running suggested code.
- Test small known answers and edge cases, then audit the real sample.
- Separate computational correctness from statistical interpretation and privacy decisions.

## Why it matters

An AI coding assistant can suggest a plausible command that uses the wrong denominator, drops observations or invents an API. Fluent explanations do not test the code. The candidate snippets here are **invented teaching examples**, not transcripts from a particular product or a model evaluation. We use made-up numbers to find a bug, then verify the real income–life expectancy regression against explicit requirements.

## Worked example

### Prepare the analysis libraries

Run this prepared cell before importing analysis packages. It uses the course dependency pins and installs only if required. If it asks for a restart, restart the notebook session and run from the top. Installation needs internet; no paid software is needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. These packages run inside the notebook session; this does not install Python on your laptop.

### Write a known answer before accepting a function

Suppose fictional Northland income rises from 100 to 150 made-up dollars. A growth rate compares the change with the starting level, so the known answer is 50%. This candidate function incorrectly divides by the ending level.

In [ ]:
def candidate_growth(start, end):
    return 100 * (end - start) / end
expected_growth = 50.0
candidate_value = candidate_growth(100, 150)
print("Candidate result:", candidate_value)
print("Expected result:", expected_growth)

The output is about 33.33%, even though the code runs without an error. A small example with a known answer can reveal a substantive mistake that a syntax check cannot find.

### Make the disagreement explicit

An assertion states a condition that must hold. This demonstration catches only the expected failed assertion for the deliberately wrong candidate.

In [ ]:
import math
try:
    assert math.isclose(candidate_value, expected_growth), "The starting-level denominator was not used."
except AssertionError as error:
    print(f"{type(error).__name__}: {error}")

Copying a candidate calculation into its own test would repeat the mistake. The expected value came from the definition and mental arithmetic, independently of the candidate implementation.

### Repair the definition and test more than one case

Require a positive starting level and divide the change by it. A valid error for zero is part of the function contract; an arbitrary zero-denominator fallback would hide an undefined rate.

In [ ]:
def growth_rate(start, end):
    if start <= 0:
        raise ValueError("Growth comparison requires a positive starting level.")
    return 100 * (end - start) / start
print(growth_rate(100, 150))
print(growth_rate(100, 100))
print(growth_rate(100, 80))

The results are 50%, 0% and -20%. These cases check increase, no change and decrease. More tests may be needed for a research application; passing three examples is evidence about those cases, not a universal guarantee.

### Ask for an inspectable analysis

A useful request names the **question, unit, fields, sample, transformation and outputs**. For example: “Using the documented historical CSV, select complete 2007 country observations. Regress life expectancy in years on the natural log of positive GDP per person, with an intercept and HC1 t-based approximate intervals. Report N and units, then explain what this association cannot identify.” Ask the assistant to state assumptions and cite the official API documentation.

Before running a suggestion, read it for writes, downloads, uploads, package installations and any step that changes inputs. Use a copy or temporary output folder. Never paste private student records, personal identifiers, passwords or restricted research data into an assistant because a code example asks for them. Provide column names and fictional records when those are sufficient. Whether a tool is permitted for a real dataset depends on the data owner's rules and the institution's approved services.

Prefer an explanation you can check over a long opaque script. Divide the task into loading, sample preparation, transformations, estimation and export. Require checks at each boundary. An invented citation or method name is not evidence; inspect the official source. Human responsibility for the research question, rights and interpretation remains even when the computation was assisted.

### Prepare the historical country panel

This prepared cell uses the local course CSV or downloads its checksum-verified public source in a fresh browser session. A checksum identifies exact file contents; changed bytes stop the preparation.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. This is a historical teaching extract: 142 countries, twelve five-year observations from 1952 to 2007. Its GDP measure is inflation-adjusted source dollars with no documented base year. It is not current statistics.

### Audit the actual sample before fitting

Spell out the 2007 sample, required columns, complete-case rule, positive-income condition and unique-country check. Here the source contains no missing values in these fields, so all 142 countries remain.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
countries = pd.read_csv(data_file)
selected = countries.loc[countries["year"] == 2007, ["country", "life_exp", "gdp_pc"]].copy()
print("Rows before complete-case rule:", len(selected))
print(selected.isna().sum())
sample = selected.dropna().copy()
assert not sample["country"].duplicated().any()
assert sample["gdp_pc"].gt(0).all()
sample["log_gdp_pc"] = np.log(sample["gdp_pc"])
print("Final countries:", len(sample))

These checks protect the unit and transformation, not just the final row count. A join could duplicate one country while dropping another and still leave 142 rows; inspect keys and sample decisions together.

### Inspect the fitted object and interpretation

Fit the model and read the formula, uncertainty setting and sample size back from the result rather than trusting a surrounding description.

In [ ]:
model = smf.ols("life_exp ~ log_gdp_pc", sample, missing="raise").fit(cov_type="HC1", use_t=True)
print("Formula:", model.model.formula)
print("Covariance:", model.cov_type)
print("N:", int(model.nobs))
print("Log-income coefficient:", model.params["log_gdp_pc"])
print("Fitted doubling difference, years:", model.params["log_gdp_pc"] * np.log(2))

Expect HC1, N=142 and about 4.993 years for the fitted doubling comparison. These are country-level associations. A statement such as “doubling a person’s income adds five years of life” would change both the unit and the causal claim. Correct syntax and a verified coefficient do not justify that statement.

### Keep a compact, useful handoff

Record the input version and rights, sample rules, package versions, rerun command, actual checks, and remaining limits. Use Git commits for exact changes and a concise README for another person; do not publish a transcript containing private prompts or data. Clearly distinguish **passed**, **failed** and **not run**. If an assistant says it tested a script, ask for the command and inspect the result you can reproduce.

For a notebook, restart and run all cells. For a script, run it in a fresh process from documented inputs, as in lesson 5.2. Compare key results with a simpler calculation or an independent implementation when that addresses a real risk. A result that changes after a legitimate data revision or specification change needs explanation, not an instruction to force it back to a preferred answer.

## Try it

Work in your own copy. Add a cell for each task and attempt it before reading a solution.

### 1. Check an increase using a second known answer.

<details><summary>Solution 1</summary>

Run this solution after your own attempt.

In [ ]:
assert math.isclose(growth_rate(200, 220), 10.0)
print("Known 10% increase passed.")

The expected 10% comes from 20/200, not from copying the candidate function. A tolerance avoids treating harmless floating-point representation as a different result.

</details>

### 2. Verify the model contract in the fitted object.

<details><summary>Solution 2</summary>

Run this solution after your own attempt.

In [ ]:
assert int(model.nobs) == 142
assert model.cov_type == "HC1"
assert model.model.formula == "life_exp ~ log_gdp_pc"
print("Sample, covariance and formula match the stated request.")

These checks confirm selected properties of this run. They do not independently prove that the model is appropriate or that cross-country errors are independent.

</details>

### 3. Check the formula against a simpler independent slope calculation.

<details><summary>Solution 3</summary>

Run this solution after your own attempt.

In [ ]:
x = sample["log_gdp_pc"].to_numpy()
y = sample["life_exp"].to_numpy()
manual_slope = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
print(manual_slope)
print(np.isclose(manual_slope, model.params["log_gdp_pc"]))

The point estimates agree for a simple OLS model with an intercept. This check does not validate the HC1 covariance formula, the source measurements or any causal interpretation.

</details>

## When it goes wrong

These demonstrations catch a specific intended error so the notebook continues. Unexpected errors elsewhere should still stop execution.

### A zero baseline makes the requested rate undefined

Test that the repaired function refuses it.

In [ ]:
try:
    growth_rate(0, 150)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Specify what invalid inputs mean before asking a tool to handle them. Returning 0% would be a new substantive convention, not a neutral fix.

### A plausible attribute is invented

This result object does not provide an attribute with the guessed name.

In [ ]:
try:
    model.standard_errors
except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

The documented statsmodels attribute is bse. Other packages use different names. Verify official documentation instead of installing extra libraries to make a fabricated API name seem real.

## Recap

- Write the question and observable checks before accepting suggested code.
- Use known small answers, edge cases and explicit sample/key checks.
- Read fitted objects and exports rather than trusting prose about them.
- Correct computation does not establish causation, permission to share data or appropriate research design.

**Next:** [Part 6 guided projects](../../projects/index.qmd). The full curriculum is drafted; publication and a first-release review remain separate steps.